In [2]:
function IR(A, b, x0; ul = Float16, u = Float32, ur = Float64, imax=10, atol=1e-6)

    x = x0

    for i in 1:imax
        # Step 1: compute residual in ur precision
        r = ur.(b) - ur.(A) * ur.(x)

        # Step 2 (low-precision inner solve in Float16 with SR):
        d16 = solve_fp16_SR(A, r; T=Float16, iters=8)  # <— SR happens inside
        
        # Step 3: update x in u precision
        x = u.(x) + u.(d)
        
        if abs(norm(r)) < atol
            println("Converged at iter $(i)")
            break
        end
    end

    return x
end

IR (generic function with 1 method)

In [4]:
function example(n, m; ul = Float16, u = Float32, ur = Float64, imax=10, atol=1e-6)

    A = rand(n, m)
    b = rand(n)
    x0 = ones(m)

    start_time = time()
    x_true = A \ b
    elapsed = time() - start_time
    x_true = vec(x_true)
    println("Run time: ", elapsed)

    start_time = time()
    x = IR(A, b, x0)
    elapsed = time() - start_time
    println("Run time: ", elapsed)
    
    error = abs(norm(x-x_true))
    println("Error: ", error)
    

end    

example (generic function with 1 method)

In [34]:
# import Pkg; Pkg.add("CUDA")
using LinearAlgebra
using BenchmarkTools
using CUDA

example(100, 100)

Run time: 0.00019598007202148438
Run time: 0.015617847442626953
Error: NaN
